# Update 1 · Evolución del notebook 04

**Primera parte:** se conservan las 31 celdas del original.

**Segunda parte, empezar en L1:** tres periodos PQRS completos; lectura y conversión CSV → Parquet con Polars; top municipal y departamentos por periodo; top municipal por grupo etario y género. Comparación pandas CPU, Polars CPU y Polars GPU.

La GPU se considera preparada. El laboratorio no se ejecutó al editarlo.


# P04 e I01. Rendimiento y traza finita

P04: clase 7, 23 de octubre; I01: clase 8, 24 de octubre de 2026.
Preparado por el PhD Esteban Hernández, CyberColombia.

[Enunciado P04](../../talleres/P04.md) · [I01](../../talleres/I01.md) · [WSL](../../docs/instalacion-wsl.md) · [Linux/macOS](../../docs/instalacion-linux-macos.md)

El viernes el profesor demuestra; el sábado los estudiantes ejecutan y entregan en clase. Ejecutar las celdas en orden con el kernel del único `.venv`. Cada lectura, consulta y escritura queda visible. No se importan scripts del curso.

## 1. Preparar y localizar los datos
Desde la terminal en `kit/`, con `.venv` activo:
```bash
python 00_datos.py --descargar pqrs nasa
python 00_datos.py --verificar pqrs nasa
```
Primero ejecutar [02 Parquet y contratos](02_Parquet_y_contratos.ipynb): allí se leen los CSV, se convierten los tipos y se escriben el CSV proyectado, el Parquet y sus particiones. Aquí se leen esos productos; no se regeneran silenciosamente. Las muestras contienen 3.000 reportes; no permiten concluir sobre rendimiento nacional.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# Editar esta ruta para que coincida con el clon local.
# Linux/macOS: Path.home() / "bigdata/big-data-postgraduate"
repositorio = Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
KIT = repositorio / "kit"
RAW = KIT / "data/raw"
OUT = KIT / "salidas" / "pandas/P04"
assert KIT.is_dir(), "Corrige repositorio antes de continuar"
OUT.mkdir(parents=True, exist_ok=True)


In [2]:
entrada = KIT / "salidas/pandas/P02"
archivo_parquet = entrada / "pqrs.parquet"
archivo_csv = entrada / "pqrs_proyeccion.csv"
particiones = entrada / "pqrs_particionado"
assert archivo_parquet.is_file(), "Ejecuta el notebook 02"
assert archivo_csv.is_file() and list(particiones.glob("**/*.parquet"))
control_entrada = json.loads((entrada / "control_formatos_pandas.json").read_text())
assert control_entrada["filas"] == 3000, "Este bloque mide las muestras de P02"
columnas = ["corte_archivo", "anio", "mes_num", "afec_cod_depto"]
# Importar antes de medir: excluimos el coste de importación de las bibliotecas.
import polars as pl
import duckdb
from time import perf_counter
import platform
mediciones = []
display(pd.DataFrame([{"python": platform.python_version(), "pandas": pd.__version__,
                       "polars": pl.__version__, "duckdb": duckdb.__version__,
                       "sistema": platform.platform()}]))


,python,pandas,polars,duckdb,sistema
0,3.12.15,2.2.3,1.35.2,1.4.1,Linux-6.18.54.1-microsoft-standard-WSL2-x86_64...


## 2. Comparar la misma consulta en cinco variantes
Agrupar **todos los años** por corte, año, mes y departamento, incluyendo claves nulas, y contar filas. Cada celda mide lectura + agregación + obtención del resultado en pandas. `%%time` informa CPU y **Wall time**; `perf_counter` guarda el tiempo transcurrido del bloque para el informe. Polars incluye `to_pandas`; DuckDB incluye la apertura/cierre de conexión.

La ordenación, normalización de tipos, comprobación y visualización se realizan después y quedan fuera del intervalo registrado. Ejecutar una vuelta de calentamiento, vaciar `mediciones` con la celda indicada y realizar tres vueltas, alternando el orden de los motores. No volver a ejecutar la configuración entre vueltas. La caché del sistema puede persistir; no llamar a esto una prueba con caché fría. Este ejercicio comparte el proceso del kernel y no mide RSS ni arranque de procesos.


### pandas: lectura inmediata y groupby


In [3]:
%%time
inicio = perf_counter()
datos_pandas = pd.read_parquet(archivo_parquet, columns=columnas)
resultado_pandas = (datos_pandas.groupby(columnas, dropna=False)
    .size().reset_index(name="reportes"))
segundos = perf_counter() - inicio
mediciones.append({"motor": "pandas", "segundos": segundos})


CPU times: user 84.3 ms, sys: 5.24 ms, total: 89.6 ms
Wall time: 489 ms


### Polars: lectura diferida y ejecución explícita
`scan_parquet` prepara el plan; `collect` lo ejecuta. Medir solo la creación de `consulta` no mide el procesamiento.


In [4]:
%%time
inicio = perf_counter()
consulta = (pl.scan_parquet(archivo_parquet)
    .group_by(columnas).agg(pl.len().alias("reportes")))
resultado_polars = consulta.collect(engine="streaming").to_pandas()
segundos = perf_counter() - inicio
mediciones.append({"motor": "polars", "segundos": segundos})


CPU times: user 6.88 ms, sys: 98.7 ms, total: 106 ms
Wall time: 80.5 ms


### DuckDB: SQL sobre Parquet
Las tres variantes siguientes usan exactamente el mismo SQL. El CSV se lee como texto para conservar los ceros de los códigos; por ello incluye un coste distinto de interpretación del formato.


In [5]:
sql = """
SELECT corte_archivo, anio, mes_num, afec_cod_depto, COUNT(*) AS reportes
FROM reportes
GROUP BY corte_archivo, anio, mes_num, afec_cod_depto
"""


### duckdb


In [6]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_parquet(str(archivo_parquet)).create_view("reportes")
    resultado_duckdb = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb", "segundos": segundos})


CPU times: user 22.3 ms, sys: 45.8 ms, total: 68.1 ms
Wall time: 317 ms


### duckdb_particionado


In [7]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_parquet(str(particiones / "**/*.parquet"), hive_partitioning=True).create_view("reportes")
    resultado_duckdb_particionado = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb_particionado", "segundos": segundos})


CPU times: user 80.2 ms, sys: 105 ms, total: 185 ms
Wall time: 3.64 s


### duckdb_csv


In [8]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_csv(str(archivo_csv), header=True, all_varchar=True).create_view("reportes")
    resultado_duckdb_csv = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb_csv", "segundos": segundos})


CPU times: user 40.6 ms, sys: 63.6 ms, total: 104 ms
Wall time: 258 ms


## 3. Verificar antes de comparar tiempos
El CSV devuelve claves numéricas como texto; armonizamos tipos fuera de la medición. No basta con comparar el total: cada grupo debe coincidir. Inspeccionar primero los tipos originales. Repetir esta comprobación después de cada vuelta.


In [9]:
resultados = {"pandas": resultado_pandas, "polars": resultado_polars,
              "duckdb": resultado_duckdb, "duckdb_particionado": resultado_duckdb_particionado,
              "duckdb_csv": resultado_duckdb_csv}
comparables = {}
for motor, resultado in resultados.items():
    print(motor, resultado.dtypes.to_dict())
    normalizado = resultado.copy()
    for columna in ["anio", "mes_num", "reportes"]:
        normalizado[columna] = pd.to_numeric(normalizado[columna], errors="raise").astype("Int64")
    normalizado["corte_archivo"] = normalizado["corte_archivo"].astype("string")
    normalizado["afec_cod_depto"] = normalizado["afec_cod_depto"].astype("string").str.zfill(2)
    comparables[motor] = normalizado.sort_values(columnas).reset_index(drop=True)
    assert normalizado["reportes"].sum() == 3000
for motor, resultado in comparables.items():
    pd.testing.assert_frame_equal(comparables["pandas"], resultado)
display(comparables["pandas"])
print("Los cinco resultados coinciden grupo por grupo")


pandas {'corte_archivo': dtype('O'), 'anio': Int64Dtype(), 'mes_num': Int64Dtype(), 'afec_cod_depto': dtype('O'), 'reportes': dtype('int64')}
polars {'corte_archivo': dtype('O'), 'anio': dtype('int64'), 'mes_num': dtype('int64'), 'afec_cod_depto': dtype('O'), 'reportes': dtype('uint32')}
duckdb {'corte_archivo': dtype('O'), 'anio': dtype('int64'), 'mes_num': dtype('int64'), 'afec_cod_depto': dtype('O'), 'reportes': dtype('int64')}
duckdb_particionado {'corte_archivo': dtype('O'), 'anio': dtype('int64'), 'mes_num': dtype('int64'), 'afec_cod_depto': dtype('O'), 'reportes': dtype('int64')}
duckdb_csv {'corte_archivo': dtype('O'), 'anio': dtype('O'), 'mes_num': dtype('O'), 'afec_cod_depto': dtype('O'), 'reportes': dtype('int64')}


,corte_archivo,anio,mes_num,afec_cod_depto,reportes
0,2023_II,2023,7,05,169
1,2023_II,2023,7,08,52
2,2023_II,2023,7,11,230
3,2023_II,2023,7,13,17
4,2023_II,2023,7,15,21
...,...,...,...,...,...
138,2024_II,2024,7,76,120
139,2024_II,2024,7,81,10
140,2024_II,2024,7,85,14
141,2024_II,2024,7,86,5


Los cinco resultados coinciden grupo por grupo


### Excluir el calentamiento
Ejecutar la siguiente celda **una vez**, tras verificar la primera vuelta. Después repetir únicamente las cinco celdas medidas y la comprobación anterior tres veces; alternar su orden. No ejecutar de nuevo esta celda al finalizar.


In [10]:
mediciones = []


### Guardar las tres repeticiones
Ejecutar tras completar las tres vueltas. Este archivo documenta el alcance del kernel; no sustituye las mediciones de RSS en procesos independientes.


In [14]:
tiempos = pd.DataFrame(mediciones)
assert len(tiempos) == 15, "Faltan las tres vueltas de cinco motores después del calentamiento"
assert tiempos.groupby("motor").size().eq(3).all()
tiempos["repeticion"] = tiempos.groupby("motor").cumcount() + 1
tiempos["alcance"] = "kernel: lectura, agregación y resultado pandas; 3000 filas"
resumen = tiempos.groupby("motor")["segundos"].agg(["median", "min", "max"]).reset_index()
tiempos.to_csv(OUT / "benchmark_notebook.csv", index=False)
resumen.to_csv(OUT / "benchmark_notebook_resumen.csv", index=False)
comparables["pandas"].to_csv(OUT / "conteos_verificados.csv", index=False)
display(resumen)


AssertionError: Faltan las tres vueltas de cinco motores después del calentamiento

## 4. Escala real y automatización, después de comprender las consultas
[P04](../../talleres/P04.md) conserva la extensión de cinco variantes en procesos independientes, dos hilos, calentamiento, tres repeticiones y sondeo RSS cada 10 ms. Revisar `trabajador` y `benchmark` en `kit/pqrs_talleres.py`: identificar en el código cada consulta que acabas de ejecutar antes de usar la automatización.

Los tiempos del CLI incluyen arranque y exportación; **no mezclar** esas mediciones con las del kernel. El CLI escribe `benchmark_pqrs.csv` y su resumen; el notebook usa otros nombres para distinguir el alcance. Registrar máquina, versiones e hilos. Las bibliotecas del kernel pueden usar distintos números de hilos: esta comparación describe la configuración local, no aísla el efecto del motor.

El volumen completo se descarga explícitamente según P04; reservar al menos 8 GB de disco más el entorno. Con 8 GB de RAM mantener pandas en muestras. Los completos concilian 2.444.766 reportes. No cambiar la ruta de este bloque de muestras a los completos sin revisar su preparación y memoria.

**Detenerse aquí en la clase 7.** Retomar eventos en la clase 8.

## 5. Leer las observaciones NASA
`json.load` lee el documento; `DataFrame` convierte el diccionario de parámetros en filas. Convertimos las fechas y separamos los faltantes (-999), sin inventar lluvia. La fecha diaria se usa como tiempo de evento por convención didáctica.


In [15]:
with open(RAW / "nasa.json", encoding="utf-8") as archivo:
    nasa = json.load(archivo)
lluvia = pd.Series(nasa["properties"]["parameter"]["PRECTOTCORR"], name="lluvia_mm")
observaciones = lluvia.rename_axis("event_id").reset_index()
observaciones["fecha_evento"] = pd.to_datetime(observaciones["event_id"], format="%Y%m%d")
observaciones["lluvia_mm"] = pd.to_numeric(observaciones["lluvia_mm"], errors="raise").replace(-999, float("nan"))
assert observaciones["event_id"].is_unique
display(observaciones.head(12))


,event_id,lluvia_mm,fecha_evento
0,20250101,1.96,2025-01-01
1,20250102,13.92,2025-01-02
2,20250103,15.27,2025-01-03
3,20250104,7.43,2025-01-04
4,20250105,9.97,2025-01-05
5,20250106,2.00,2025-01-06
6,20250107,0.50,2025-01-07
7,20250108,0.28,2025-01-08
8,20250109,0.07,2025-01-09
9,20250110,0.01,2025-01-10


## 6. Construir el orden de entrega y unir los valores reales
Primero llegan los días 1–4 y 6–12; luego se reintenta el 6 y finalmente llega el 5. `merge(validate="many_to_one")` permite entregas repetidas pero exige una observación por fecha. Inspeccionar la tabla antes de aplicar reglas.


In [16]:
dias = [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 6, 5]
entregas = pd.DataFrame({"event_id": [f"202501{dia:02d}" for dia in dias]})
entregas["orden_entrega"] = range(1, len(entregas) + 1)
entregas = entregas.merge(observaciones, on="event_id", how="left", validate="many_to_one", indicator=True)
assert entregas["_merge"].eq("both").all() and entregas["lluvia_mm"].notna().all()
display(entregas)


,event_id,orden_entrega,lluvia_mm,fecha_evento,_merge
0,20250101,1,1.96,2025-01-01,both
1,20250102,2,13.92,2025-01-02,both
2,20250103,3,15.27,2025-01-03,both
3,20250104,4,7.43,2025-01-04,both
4,20250106,5,2.00,2025-01-06,both
5,20250107,6,0.50,2025-01-07,both
6,20250108,7,0.28,2025-01-08,both
7,20250109,8,0.07,2025-01-09,both
8,20250110,9,0.01,2025-01-10,both
9,20250111,10,0.00,2025-01-11,both


## 7. Recorrer la traza paso a paso
Antes de cada entrega, el límite es la fecha máxima observada menos tres días. Primero comprobamos si el identificador ya fue aceptado; después, si la fecha es anterior al límite. Una fecha igual al límite se admite. La lista `traza` conserva la decisión y el límite usado; `vistos` solo guarda aceptados. Esta regla finita no reproduce toda la semántica de Spark.


In [17]:
vistos = set()
ultima_fecha = None
traza = []
for entrega in entregas.itertuples(index=False):
    limite = ultima_fecha - pd.Timedelta(days=3) if ultima_fecha is not None else None
    if entrega.event_id in vistos:
        estado = "duplicado"
    elif limite is not None and entrega.fecha_evento < limite:
        estado = "tardio"
    else:
        estado = "aceptado"
        vistos.add(entrega.event_id)
    traza.append({"orden_entrega": entrega.orden_entrega, "event_id": entrega.event_id,
                  "fecha_evento": entrega.fecha_evento, "lluvia_mm": entrega.lluvia_mm,
                  "watermark_antes": limite, "estado": estado})
    ultima_fecha = max(ultima_fecha, entrega.fecha_evento) if ultima_fecha is not None else entrega.fecha_evento
    print(entrega.orden_entrega, entrega.event_id, "límite:", limite, estado)
traza = pd.DataFrame(traza)


1 20250101 límite: None aceptado
2 20250102 límite: 2024-12-29 00:00:00 aceptado
3 20250103 límite: 2024-12-30 00:00:00 aceptado
4 20250104 límite: 2024-12-31 00:00:00 aceptado
5 20250106 límite: 2025-01-01 00:00:00 aceptado
6 20250107 límite: 2025-01-03 00:00:00 aceptado
7 20250108 límite: 2025-01-04 00:00:00 aceptado
8 20250109 límite: 2025-01-05 00:00:00 aceptado
9 20250110 límite: 2025-01-06 00:00:00 aceptado
10 20250111 límite: 2025-01-07 00:00:00 aceptado
11 20250112 límite: 2025-01-08 00:00:00 aceptado
12 20250106 límite: 2025-01-09 00:00:00 duplicado
13 20250105 límite: 2025-01-09 00:00:00 tardio


## 8. Conciliar y escribir las evidencias
Comparar entregas, fechas distintas y aceptados. Explicar qué observación falta en la suma aceptada y proponer una corrección antes de continuar con Spark en E11.


In [18]:
aceptados = traza.loc[traza["estado"].eq("aceptado")]
control = {"entregas": len(traza), "registros_reales_distintos": int(traza["event_id"].nunique()),
           "aceptados": len(aceptados), "duplicados": int(traza["estado"].eq("duplicado").sum()),
           "tardios": int(traza["estado"].eq("tardio").sum()),
           "suma_aceptada_mm": float(aceptados["lluvia_mm"].sum()),
           "suma_fechas_unicas_mm": float(entregas.drop_duplicates("event_id")["lluvia_mm"].sum())}
assert (control["entregas"], control["registros_reales_distintos"], control["aceptados"], control["duplicados"], control["tardios"]) == (13, 12, 11, 1, 1)
assert abs(control["suma_aceptada_mm"] - 41.44) < 1e-8
traza.to_csv(OUT / "traza_eventos_nasa.csv", index=False)
with open(OUT / "control_eventos_intro.json", "w", encoding="utf-8") as archivo:
    json.dump(control, archivo, ensure_ascii=False, indent=2)
recuperada = pd.read_csv(OUT / "traza_eventos_nasa.csv", dtype={"event_id": "string"},
                         parse_dates=["fecha_evento", "watermark_antes"])
pd.testing.assert_frame_equal(traza.astype({"event_id": "string"}), recuperada)
display(control)
display(recuperada)


{'entregas': 13,
 'registros_reales_distintos': 12,
 'aceptados': 11,
 'duplicados': 1,
 'tardios': 1,
 'suma_aceptada_mm': 41.44,
 'suma_fechas_unicas_mm': 51.41}

,orden_entrega,event_id,fecha_evento,lluvia_mm,watermark_antes,estado
0,1,20250101,2025-01-01,1.96,NaT,aceptado
1,2,20250102,2025-01-02,13.92,2024-12-29,aceptado
2,3,20250103,2025-01-03,15.27,2024-12-30,aceptado
3,4,20250104,2025-01-04,7.43,2024-12-31,aceptado
4,5,20250106,2025-01-06,2.00,2025-01-01,aceptado
5,6,20250107,2025-01-07,0.50,2025-01-03,aceptado
6,7,20250108,2025-01-08,0.28,2025-01-04,aceptado
7,8,20250109,2025-01-09,0.07,2025-01-05,aceptado
8,9,20250110,2025-01-10,0.01,2025-01-06,aceptado
9,10,20250111,2025-01-11,0.00,2025-01-07,aceptado


## Entrega
Adjuntar tiempos, resultados equivalentes, configuración y alcance de la medición; distinguir notebook y CLI. Entregar la traza y explicar el reintento y el atraso. Modificar una condición de consulta, anticipar el resultado y verificarlo en todos los motores. No sumar lluvia con PQRS ni fabricar tiempos de evento para reportes.


# Laboratorio P04 · Municipios con más quejas: pandas y Polars GPU

Con la GPU ya verificada, descargaremos tres CSV completos, los leeremos y
convertiremos a Parquet **con Polars** y resolveremos dos preguntas:

1. ¿Cuáles son los municipios con más quejas en cada periodo y a qué departamentos pertenecen?
2. ¿Cuáles son los municipios con más quejas por grupo etario y género en cada periodo?

Usaremos 2023-II, 2024-I y 2024-II: 2.444.766 reportes de referencia.
Una fila cuenta como reporte, no como persona única; no deduplicamos.
Usaremos `afec_cod_depto`, `afec_cod_mpio`, `afec_edad` y `afec_genero`:
ubicación y demografía de la **persona afectada**, no del peticionario ni la entidad.

**Empieza en L1 y ejecuta en orden.** Aquí están descarga, conversión y consultas.
Los tres motores analizarán los mismos Parquet. El top 10 será configurable.


## L1. Indicar la carpeta del repositorio

**Haz esto:** ejecuta la celda siguiente. Si no encuentra el repositorio,
cambia `REPOSITORIO = None` por la ruta de tu clon y vuelve a ejecutarla.
En WSL, una ruta de Windows empieza por `/mnt/c/`, no por `C:\`.

Deja los tres periodos indicados. La celda crea estas carpetas si no existen:

| Carpeta | Qué contendrá |
|---|---|
| `kit/data/pqrs_completos/` | Los tres CSV descargados. |
| `kit/salidas/pandas/P04_update1/parquet/` | Un Parquet y su control por periodo. |
| `kit/salidas/pandas/P04_update1/` | Resultados del análisis y tiempos. |

**Comprueba:** deben aparecer tres periodos y el total esperado. Reserva espacio
para los CSV, los Parquet y los temporales. `BLOQUE = 50_000` indica cuántas
filas procesará pandas en cada lote; no limita toda la memoria del proceso.


In [19]:
from pathlib import Path
from time import perf_counter
from importlib import metadata
from IPython.display import display
import csv, hashlib, json, os, platform, shutil, subprocess, sys
import urllib.request
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

REPOSITORIO = None  # Ejemplo WSL: Path('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate')
PERIODOS = ['2023_II', '2024_I', '2024_II']
BLOQUE = 50_000
REPETICIONES = 3
TOP_N = 10
DISPOSITIVO_CUDA = 0  # Índice CUDA, no el número de GPU del Administrador de tareas
if REPOSITORIO is None:
    REPOSITORIO = next((p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / 'kit/pqrs_fuentes.json').is_file()), None)
assert REPOSITORIO is not None, 'Configurar REPOSITORIO'
REPOSITORIO = Path(REPOSITORIO).expanduser().resolve()
KIT = REPOSITORIO / 'kit'
CSV_DIR = KIT / 'data/pqrs_completos'
OUT = KIT / 'salidas/pandas/P04_update1'
PARQUET_DIR = OUT / 'parquet'
for carpeta in [CSV_DIR, OUT, PARQUET_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)
COLUMNAS_CSV = ['OBJECTID',
 'periodo',
 'mes',
 'pqr_canal',
 'pet_cod_depto',
 'pet_cod_mpio',
 'id_afec',
 'afec_parentesco',
 'afec_genero',
 'afec_edad',
 'afec_edadr',
 'afec_educ',
 'afec_regafiliacion',
 'afec_getnico',
 'afec_pobespecial',
 'afec_cod_depto',
 'afec_cod_mpio',
 'ent_nombre',
 'ent_tipovig_sns',
 'ent_cod_sns',
 'ent_alias_sn',
 'ent_cod_depto',
 'ent_cod_mpio',
 'cod_macromot',
 'macromotivo',
 'cod_motgen',
 'motivo_general',
 'cod_motesp',
 'motivo_especifico',
 'cod_tipo_motesp',
 'tipo_de_motivo_especifico',
 'cod_subtipo_motesp',
 'subtipo_de_motivo_especifico',
 'patologia_1',
 'patologia_tipo',
 'cie_10',
 'alto_costo',
 'clasificacion_de_riesgo']

FUENTES = [{'periodo': '2023_II',
  'archivo': 'PQRD_2023_II.csv',
  'url': 'https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/515d4c6366854f549b24c704320f10f4/data',
  'filas': 716697,
  'bytes': 532695991,
  'separador': ';',
  'sha256': '78c21edacfc5b60afd1b8e27cffcca41ee1c1faf65d33a410a4d792739e8aec8'},
 {'periodo': '2024_I',
  'archivo': 'PQRD_2024_I.csv',
  'url': 'https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b52330a16b3940c39d38cb164c9dc014/data',
  'filas': 781601,
  'bytes': 576710688,
  'separador': ',',
  'sha256': 'f090c0a1eee399bfa6512fbaf9451d422ac3fe0fd0789a7819e0d8df0cd5e956'},
 {'periodo': '2024_II',
  'archivo': 'PQRD_2024_II.csv',
  'url': 'https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b257c9907b754b0cabb55a32e706bdee/data',
  'filas': 946468,
  'bytes': 706836370,
  'separador': ';',
  'sha256': '03ec2dd5363673190239085c49b1e731d77ead8e0356817a2b34c44765f99f3d'}]
for fuente in FUENTES:
    fuente['columnas'] = COLUMNAS_CSV
    fuente['codificacion'] = 'utf-8-sig'
assert len(FUENTES) == 3
assert [f['periodo'] for f in FUENTES] == PERIODOS
TOTAL_ESPERADO = sum(f['filas'] for f in FUENTES)
assert BLOQUE > 0 and REPETICIONES >= 3
CLAVES = ['corte_archivo', 'codigo_departamento', 'codigo_municipio']
CLAVES_DEMOGRAFICAS = ['corte_archivo', 'grupo_etario', 'genero',
                       'codigo_departamento', 'codigo_municipio']
CONSULTAS = {'municipios_periodo': CLAVES,
             'municipios_edad_genero': CLAVES_DEMOGRAFICAS}
RECETA_CONVERSION = 'polars_municipios_edad_genero_v1'
assert TOP_N >= 1
print('Intérprete:', sys.executable)
print('Versiones:', {'pandas': pd.__version__, 'polars': pl.__version__, 'pyarrow': pa.__version__})
display(pd.DataFrame([{k: f[k] for k in ['periodo', 'archivo', 'filas', 'bytes', 'separador', 'url']}
                      for f in FUENTES]))
print('Total esperado:', TOTAL_ESPERADO, 'reportes')
print('Libres en disco (GiB):', shutil.disk_usage(KIT).free / 2**30)


Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
Versiones: {'pandas': '2.2.3', 'polars': '1.35.2', 'pyarrow': '19.0.1'}


,periodo,archivo,filas,bytes,separador,url
0,2023_II,PQRD_2023_II.csv,716697,532695991,;,https://mapas.supersalud.gov.co/arcgisportal/s...
1,2024_I,PQRD_2024_I.csv,781601,576710688,",",https://mapas.supersalud.gov.co/arcgisportal/s...
2,2024_II,PQRD_2024_II.csv,946468,706836370,;,https://mapas.supersalud.gov.co/arcgisportal/s...


Total esperado: 2444766 reportes
Libres en disco (GiB): 705.9214782714844


## L2. Descargar y comprobar los tres CSV completos

Usaremos el descargador del curso. **En una terminal Ubuntu/WSL, desde la raíz
del repositorio**, entra en `kit` y ejecuta:

```bash
cd kit
python 00_datos.py --descargar pqrs-completos
python 00_datos.py --verificar pqrs-completos
```

El primer comando descarga secuencialmente **2023-II, 2024-I y 2024-II** en
`kit/data/pqrs_completos/` (aproximadamente **1,82 GB**). Si un CSV ya existe y
su huella coincide, lo conserva. Si difiere, informa el problema sin sobrescribirlo.
El segundo comando comprueba huellas, cabeceras, filas y estructura.

**Deben quedar estos archivos:**

| Periodo | Archivo | Reportes esperados |
|---|---|---:|
| 2023-II | `PQRD_2023_II.csv` | 716.697 |
| 2024-I | `PQRD_2024_I.csv` | 781.601 |
| 2024-II | `PQRD_2024_II.csv` | 946.468 |

**También puedes ejecutar directamente la celda siguiente:** invoca los mismos
comandos con el Python del kernel, sin cambiar de entorno. Si ya los ejecutaste
en terminal, conservará los archivos correctos y volverá a verificarlos.
No hace falta pegar las URL ni usar otro notebook.

**Comprueba:** deben aparecer los tres archivos con estado `coincide` y sumar
**2.444.766 reportes**. Si un comando falla, la celda se detiene. Resuelve el
problema antes de pasar a L3. Estos comandos descargan y verifican CSV;
la conversión a Parquet la harás en L4.


In [ ]:
# Ejecutar desde Jupyter los mismos comandos explicados arriba.
# sys.executable conserva el Python y el entorno del kernel actual.
subprocess.run([sys.executable, '00_datos.py', '--descargar', 'pqrs-completos'],
               cwd=KIT, check=True)
subprocess.run([sys.executable, '00_datos.py', '--verificar', 'pqrs-completos'],
               cwd=KIT, check=True)

descargas = json.loads((KIT / 'salidas/verificacion.json').read_text(encoding='utf-8'))
assert len(descargas) == 3
assert {r['archivo'] for r in descargas} == {f['archivo'] for f in FUENTES}
assert all(r['estado'] == 'coincide' for r in descargas)
assert sum(r['filas'] for r in descargas) == TOTAL_ESPERADO
print('Tres CSV completos descargados y verificados:', TOTAL_ESPERADO, 'reportes')
display(pd.DataFrame(descargas)[['archivo', 'estado', 'filas', 'bytes']])

# En L4 también comprobaremos la huella de cada Parquet sin cargarlo en RAM.
def sha256_archivo(archivo):
    huella = hashlib.sha256()
    with archivo.open('rb') as stream:
        for bloque in iter(lambda: stream.read(1024 * 1024), b''):
            huella.update(bloque)
    return huella.hexdigest()


## L3. Leer las primeras filas con Polars

Ejecuta la celda. `pl.read_csv` lee cinco filas de cada CSV con su separador y
sus columnas como texto, conservando ceros iniciales y vacíos.
Observa periodo, códigos municipales, edad y género de la persona afectada.


In [ ]:
for fuente in FUENTES:
    vista = pl.read_csv(CSV_DIR / fuente['archivo'], separator=fuente['separador'],
        infer_schema=False, missing_utf8_is_empty_string=True, n_rows=5)
    print('Corte:', fuente['periodo'], 'Columnas:', vista.width)
    display(vista.select('OBJECTID', 'periodo', 'mes', 'afec_cod_depto',
        'afec_cod_mpio', 'afec_edad', 'afec_genero'))


## L4. Convertir TODOS los CSV a Parquet con Polars

Ejecuta la celda y sigue el recorrido:

1. `scan_csv` describe la lectura y conserva las 38 columnas originales como texto.
2. `with_columns` añade corte, año y mes, y prepara claves territoriales y demográficas.
3. Normaliza departamento a dos dígitos y municipio a cinco. Un código municipal
   de hasta tres dígitos se completa con el departamento. Si es inválido o los
   cinco dígitos contradicen el departamento, se marca `SIN DATO`.
4. Construye grupo etario y normaliza género (espacios y mayúsculas), sin inferir categorías.
5. `sink_parquet` ejecuta la conversión en streaming y escribe Zstandard.
6. Comprueba filas, esquema y huella antes de continuar con el siguiente periodo.

Grupos **didácticos**, desde `afec_edad`: 0–5, 6–11, 12–17, 18–28, 29–59 y 60 o más.
Los intervalos son [0,6), [6,12), [12,18), [18,29), [29,60) y desde 60.
Solo se admiten edades finitas entre 0 y 120 años; lo demás queda `Sin dato válido`.
El campo original `afec_edadr` se conserva, sin atribuirle esta clasificación.

**La lectura y conversión CSV → Parquet se hace enteramente con Polars.**
Esta preparación usa streaming en CPU; las agregaciones se ejecutarán también
en GPU en L8. PyArrow solo inspecciona Parquet y permite lotes para el análisis pandas.
No se eliminan reportes, duplicados ni campos originales.

Un Parquet previo solo se reutiliza si coincide también la receta actual.
Los productos de la versión anterior se regenerarán al ejecutar esta celda.
Al terminar deben estar los tres periodos y sumar 2.444.766 reportes.


In [ ]:
conversiones = []
for fuente in FUENTES:
    periodo = fuente['periodo']
    parquet = PARQUET_DIR / f'{periodo}.parquet'
    control_path = PARQUET_DIR / f'{periodo}.control.json'
    esquema = {nombre: pl.String for nombre in fuente['columnas']}
    esquema.update({'corte_archivo': pl.String, 'anio': pl.Int64, 'mes_num': pl.Int64,
        'codigo_departamento': pl.String, 'codigo_municipio': pl.String,
        'grupo_etario': pl.String, 'genero': pl.String})
    valido = False
    if parquet.is_file() and control_path.is_file():
        try:
            anterior = json.loads(control_path.read_text())
            valido = (anterior.get('receta') == RECETA_CONVERSION and
                anterior.get('motor_conversion') == 'polars' and
                anterior['sha256_csv'] == fuente['sha256'] and
                anterior['filas'] == fuente['filas'] == pq.ParquetFile(parquet).metadata.num_rows and
                dict(pl.scan_parquet(parquet).collect_schema()) == esquema and
                anterior['sha256_parquet'] == sha256_archivo(parquet))
        except (OSError, ValueError, KeyError, pa.ArrowException):
            valido = False
    if valido:
        conversiones.append({**anterior, 'estado': 'reutilizado_verificado'})
        print(periodo, 'Parquet Polars ya verificado', flush=True)
        continue

    print('Convirtiendo CSV → Parquet con Polars:', periodo, flush=True)
    inicio = perf_counter()
    temporal = parquet.with_suffix('.parquet.part')
    datos = pl.scan_csv(CSV_DIR / fuente['archivo'], separator=fuente['separador'],
        infer_schema=False, missing_utf8_is_empty_string=True)
    assert datos.collect_schema().names() == fuente['columnas']
    depto = pl.col('afec_cod_depto').str.strip_chars()
    muni = pl.col('afec_cod_mpio').str.strip_chars()
    edad = pl.col('afec_edad').str.strip_chars().cast(pl.Float64, strict=False)
    genero = pl.col('afec_genero').str.strip_chars().str.to_uppercase()
    datos = datos.with_columns(
        pl.lit(periodo).alias('corte_archivo'),
        pl.col('periodo').cast(pl.Int64, strict=False).alias('anio'),
        pl.col('mes').cast(pl.Int64, strict=False).alias('mes_num'),
        pl.when(depto.str.contains(r'^\d{1,2}$'))
          .then(depto.str.pad_start(2, '0')).otherwise(pl.lit('SIN DATO'))
          .alias('codigo_departamento'),
        pl.when(~(edad.is_finite() & edad.is_between(0, 120)).fill_null(False))
          .then(pl.lit('Sin dato válido'))
          .when(edad < 6).then(pl.lit('0–5'))
          .when(edad < 12).then(pl.lit('6–11'))
          .when(edad < 18).then(pl.lit('12–17'))
          .when(edad < 29).then(pl.lit('18–28'))
          .when(edad < 60).then(pl.lit('29–59'))
          .otherwise(pl.lit('60 o más')).alias('grupo_etario'),
        pl.when(genero.is_not_null() & (genero != ''))
          .then(genero).otherwise(pl.lit('SIN DATO')).alias('genero'))
    candidato = (pl.when(muni.str.contains(r'^\d{5}$')).then(muni)
        .when(muni.str.contains(r'^\d{1,3}$') & (pl.col('codigo_departamento') != 'SIN DATO'))
        .then(pl.concat_str([pl.col('codigo_departamento'), muni.str.pad_start(3, '0')]))
        .otherwise(pl.lit('SIN DATO')))
    datos = datos.with_columns(
        pl.when(candidato.str.slice(0, 2) == pl.col('codigo_departamento'))
          .then(candidato).otherwise(pl.lit('SIN DATO')).alias('codigo_municipio'))
    datos = datos.select(list(esquema))
    assert dict(datos.collect_schema()) == esquema
    datos.sink_parquet(temporal, compression='zstd', engine='streaming')
    segundos_conversion = perf_counter() - inicio
    filas = pq.ParquetFile(temporal).metadata.num_rows
    assert filas == fuente['filas'], f'Conteo distinto en {periodo}: {filas}'
    temporal.replace(parquet)
    control = {'periodo': periodo, 'archivo': str(parquet), 'filas': filas,
        'columnas': len(esquema), 'sha256_csv': fuente['sha256'],
        'sha256_parquet': sha256_archivo(parquet),
        'bytes_csv': fuente['bytes'], 'bytes_parquet': parquet.stat().st_size,
        'segundos_conversion': segundos_conversion,
        'motor_conversion': 'polars', 'motor_ejecucion': 'streaming CPU',
        'receta': RECETA_CONVERSION}
    control_path.write_text(json.dumps(control, ensure_ascii=False, indent=2))
    conversiones.append({**control, 'estado': 'convertido_verificado'})

assert len(conversiones) == len(FUENTES)
assert sum(c['filas'] for c in conversiones) == TOTAL_ESPERADO
ARCHIVOS_PARQUET = [PARQUET_DIR / f"{f['periodo']}.parquet" for f in FUENTES]
display(pd.DataFrame(conversiones)[['periodo', 'estado', 'filas', 'columnas',
    'bytes_csv', 'bytes_parquet', 'segundos_conversion']])
print('Tres periodos convertidos con Polars y verificados. Podemos analizar.')


## L5. Los ejercicios municipales

**Ejercicio 1. Municipios con más quejas y sus departamentos, por periodo.**
Contar por corte, departamento y municipio. Construir el top `TOP_N` nacional
por corte, mostrar nombres de municipio y departamento, y resumir qué
 departamentos tienen municipios en ese top. Esto no equivale a ordenar
 departamentos por el total de todos sus reportes.

**Ejercicio 2. Municipios con más quejas por grupo etario y género.**
Añadir `grupo_etario` y `genero` a la agrupación. Construir un top separado para
**cada combinación de corte, grupo etario y género**, no un único top global.
Las categorías sin dato se conservan como tales.

En empates se usa código de departamento y municipio ascendente.
`TOP_N=10` devuelve hasta diez municipios por grupo. Solo municipios reconocidos
en DIVIPOLA entran en rankings; el resto se reporta aparte, sin perderse del total.

**Benchmark:** lectura Parquet + agregación + resultado en CPU. La unión de
nombres y los rankings sobre los agregados se hacen después, fuera del reloj,
para los tres motores por igual. Las agregaciones pesadas sí se ejecutan en GPU.


## L6. Primero: agregaciones y tiempos con pandas

Ejecuta primero la función, luego las dos celdas medidas, una por ejercicio.
Dentro de la función se ven las operaciones reales: leer lotes Parquet,
`groupby(...).size()` para contar y otro `groupby(...).sum()` para combinar
los conteos parciales. No se vuelve a leer CSV ni se convierte a Parquet.

Cada celda inicia `perf_counter`, ejecuta la consulta y detiene el reloj.
Después comprueba el total y muestra segundos y número de grupos.
Así obtenemos una referencia para las mismas consultas en Polars CPU y GPU.
El tiempo incluye lectura Parquet y agregación, no solo el `groupby` aislado.


In [ ]:
def analizar_pandas(claves):
    parciales = []
    for archivo in ARCHIVOS_PARQUET:
        for lote in pq.ParquetFile(archivo).iter_batches(batch_size=BLOQUE, columns=claves):
            datos = lote.to_pandas()
            parcial = datos.groupby(claves, dropna=False).size().reset_index(name='reportes')
            parciales.append(parcial)
    acumulados = pd.concat(parciales, ignore_index=True)
    return acumulados.groupby(claves, dropna=False, as_index=False)['reportes'].sum()

referencias = {}
primeras = []


### pandas CPU · Ejercicio 1

Contar quejas por municipio y periodo.


In [ ]:
# Inicio de la frontera medida: lectura Parquet + agregación + resultado CPU.
inicio = perf_counter()
municipios_pandas_cpu = analizar_pandas(CLAVES)
segundos = perf_counter() - inicio
# Fuera del reloj: comprobar y mostrar resultados.
assert int(municipios_pandas_cpu['reportes'].sum()) == TOTAL_ESPERADO
referencias['municipios_periodo'] = municipios_pandas_cpu
primeras = [fila for fila in primeras
    if not (fila['consulta'] == 'municipios_periodo' and fila['motor'] == 'pandas_cpu')]
primeras.append({'consulta': 'municipios_periodo', 'motor': 'pandas_cpu',
    'segundos': segundos, 'filas_entrada': TOTAL_ESPERADO,
    'grupos': len(municipios_pandas_cpu), 'equivalente': True})
print('pandas CPU · ejercicio 1:', round(segundos, 4), 's;', len(municipios_pandas_cpu), 'grupos')
display(municipios_pandas_cpu.head(10))


### pandas CPU · Ejercicio 2

Contar quejas por municipio, periodo, grupo etario y género.


In [ ]:
# Inicio de la frontera medida: lectura Parquet + agregación + resultado CPU.
inicio = perf_counter()
demografia_pandas_cpu = analizar_pandas(CLAVES_DEMOGRAFICAS)
segundos = perf_counter() - inicio
# Fuera del reloj: comprobar y mostrar resultados.
assert int(demografia_pandas_cpu['reportes'].sum()) == TOTAL_ESPERADO
referencias['municipios_edad_genero'] = demografia_pandas_cpu
primeras = [fila for fila in primeras
    if not (fila['consulta'] == 'municipios_edad_genero' and fila['motor'] == 'pandas_cpu')]
primeras.append({'consulta': 'municipios_edad_genero', 'motor': 'pandas_cpu',
    'segundos': segundos, 'filas_entrada': TOTAL_ESPERADO,
    'grupos': len(demografia_pandas_cpu), 'equivalente': True})
print('pandas CPU · ejercicio 2:', round(segundos, 4), 's;', len(demografia_pandas_cpu), 'grupos')
display(demografia_pandas_cpu.head(10))


## L7. Después: las mismas agregaciones y tiempos con Polars CPU

Ejecuta la celda que define los dos planes y, después, cada celda de medición.
Las operaciones son `scan_parquet`, `group_by` y `agg(pl.len())`.
`collect(engine='streaming')` ejecuta el trabajo en CPU.

Cronometramos la ejecución, no la creación del plan. `.to_pandas()` queda dentro
del reloj para obtener la misma representación final en los tres motores.
Las comprobaciones se hacen después de detenerlo. Observa los segundos y
comprueba igualdad de todos los grupos con pandas.


In [ ]:
# Ejercicio 1: municipios en cada periodo.
consulta_municipios = (
    pl.scan_parquet(ARCHIVOS_PARQUET)
    .select(CLAVES)
    .group_by(CLAVES)
    .agg(pl.len().alias('reportes'))
)

# Ejercicio 2: municipios por grupo etario y género en cada periodo.
consulta_demografica = (
    pl.scan_parquet(ARCHIVOS_PARQUET)
    .select(CLAVES_DEMOGRAFICAS)
    .group_by(CLAVES_DEMOGRAFICAS)
    .agg(pl.len().alias('reportes'))
)
consultas_polars = {'municipios_periodo': consulta_municipios,
                   'municipios_edad_genero': consulta_demografica}
print(consulta_municipios.explain())

def comparable(resultado, claves):
    copia = resultado.copy()
    copia['reportes'] = pd.to_numeric(copia['reportes'], errors='raise').astype('Int64')
    for nombre in claves:
        copia[nombre] = copia[nombre].astype('string')
    return copia.sort_values(claves, na_position='last').reset_index(drop=True)[claves + ['reportes']]


### Polars CPU · Ejercicio 1

Contar quejas por municipio y periodo.


In [ ]:
# Inicio de la frontera medida: lectura Parquet + agregación + resultado CPU.
inicio = perf_counter()
municipios_polars_cpu = consulta_municipios.collect(engine='streaming').to_pandas()
segundos = perf_counter() - inicio
# Fuera del reloj: comprobar y mostrar resultados.
pd.testing.assert_frame_equal(
    comparable(referencias['municipios_periodo'], CLAVES),
    comparable(municipios_polars_cpu, CLAVES))
primeras = [fila for fila in primeras
    if not (fila['consulta'] == 'municipios_periodo' and fila['motor'] == 'polars_cpu')]
primeras.append({'consulta': 'municipios_periodo', 'motor': 'polars_cpu',
    'segundos': segundos, 'filas_entrada': TOTAL_ESPERADO,
    'grupos': len(municipios_polars_cpu), 'equivalente': True})
print('Polars CPU · ejercicio 1:', round(segundos, 4), 's;', len(municipios_polars_cpu), 'grupos')
display(municipios_polars_cpu.head(10))


### Polars CPU · Ejercicio 2

Contar quejas por municipio, periodo, grupo etario y género.


In [ ]:
# Inicio de la frontera medida: lectura Parquet + agregación + resultado CPU.
inicio = perf_counter()
demografia_polars_cpu = consulta_demografica.collect(engine='streaming').to_pandas()
segundos = perf_counter() - inicio
# Fuera del reloj: comprobar y mostrar resultados.
pd.testing.assert_frame_equal(
    comparable(referencias['municipios_edad_genero'], CLAVES_DEMOGRAFICAS),
    comparable(demografia_polars_cpu, CLAVES_DEMOGRAFICAS))
primeras = [fila for fila in primeras
    if not (fila['consulta'] == 'municipios_edad_genero' and fila['motor'] == 'polars_cpu')]
primeras.append({'consulta': 'municipios_edad_genero', 'motor': 'polars_cpu',
    'segundos': segundos, 'filas_entrada': TOTAL_ESPERADO,
    'grupos': len(demografia_polars_cpu), 'equivalente': True})
print('Polars CPU · ejercicio 2:', round(segundos, 4), 's;', len(demografia_polars_cpu), 'grupos')
display(demografia_polars_cpu.head(10))


## L8. Finalmente: las mismas agregaciones y tiempos con Polars GPU

La GPU ya está verificada. Ejecuta las dos celdas siguientes en su kernel.
El plan de agregación es el mismo que en Polars CPU; cambia el motor en `collect`.
`raise_on_fail=True` impide registrar un fallback a CPU como ejecución GPU.

Cada celda muestra segundos, grupos y equivalencia. El resultado vuelve a CPU;
esa devolución y `.to_pandas()` están dentro del tiempo. No es una medición
exclusiva de kernels CUDA ni de datos ya residentes en VRAM.

Al terminar, la tabla mostrará los tres tiempos para cada ejercicio. Estas son
primeras ejecuciones: después repetiremos para comparar medianas.


In [ ]:
engine_gpu = pl.GPUEngine(device=DISPOSITIVO_CUDA, raise_on_fail=True)
gpu_verificada = False


### Polars GPU · Ejercicio 1

Contar quejas por municipio y periodo.


In [ ]:
# Inicio de la frontera medida: lectura Parquet + agregación + resultado CPU.
inicio = perf_counter()
municipios_polars_gpu = consulta_municipios.collect(engine=engine_gpu).to_pandas()
segundos = perf_counter() - inicio
# Fuera del reloj: comprobar y mostrar resultados.
pd.testing.assert_frame_equal(
    comparable(referencias['municipios_periodo'], CLAVES),
    comparable(municipios_polars_gpu, CLAVES))
primeras = [fila for fila in primeras
    if not (fila['consulta'] == 'municipios_periodo' and fila['motor'] == 'polars_gpu')]
primeras.append({'consulta': 'municipios_periodo', 'motor': 'polars_gpu',
    'segundos': segundos, 'filas_entrada': TOTAL_ESPERADO,
    'grupos': len(municipios_polars_gpu), 'equivalente': True})
print('Polars GPU · ejercicio 1:', round(segundos, 4), 's;', len(municipios_polars_gpu), 'grupos')
display(municipios_polars_gpu.head(10))

municipios_gpu = municipios_polars_gpu


### Polars GPU · Ejercicio 2

Contar quejas por municipio, periodo, grupo etario y género.


In [ ]:
# Inicio de la frontera medida: lectura Parquet + agregación + resultado CPU.
inicio = perf_counter()
demografia_polars_gpu = consulta_demografica.collect(engine=engine_gpu).to_pandas()
segundos = perf_counter() - inicio
# Fuera del reloj: comprobar y mostrar resultados.
pd.testing.assert_frame_equal(
    comparable(referencias['municipios_edad_genero'], CLAVES_DEMOGRAFICAS),
    comparable(demografia_polars_gpu, CLAVES_DEMOGRAFICAS))
primeras = [fila for fila in primeras
    if not (fila['consulta'] == 'municipios_edad_genero' and fila['motor'] == 'polars_gpu')]
primeras.append({'consulta': 'municipios_edad_genero', 'motor': 'polars_gpu',
    'segundos': segundos, 'filas_entrada': TOTAL_ESPERADO,
    'grupos': len(demografia_polars_gpu), 'equivalente': True})
print('Polars GPU · ejercicio 2:', round(segundos, 4), 's;', len(demografia_polars_gpu), 'grupos')
display(demografia_polars_gpu.head(10))

demografia_gpu = demografia_polars_gpu
gpu_verificada = True


### Comparación de las primeras ejecuciones

Las filas son los ejercicios y las columnas son motores. Todos usan los mismos
Parquet, claves y reportes. El menor tiempo es el mejor en esta ejecución.
No concluimos todavía con una sola medición: L9 obtiene tres repeticiones.


In [ ]:
tabla_primeras = pd.DataFrame(primeras)
assert len(tabla_primeras) == 6, 'Completar los dos ejercicios en los tres motores'
comparacion_inicial = tabla_primeras.pivot(index='consulta', columns='motor', values='segundos')
comparacion_inicial = comparacion_inicial[['pandas_cpu', 'polars_cpu', 'polars_gpu']]
print('Segundos de primera ejecución:')
display(comparacion_inicial)
comparacion_inicial.to_csv(OUT / 'comparacion_primera_ejecucion.csv')


## L9. Repetir y comparar el rendimiento

**Haz esto:** después de las seis mediciones visibles de L6–L8, ejecuta la celda. Hará un calentamiento
adicional y **tres repeticiones por consulta y motor**, alternando el orden.
Con dos consultas y tres motores, al final deben existir **18 mediciones**.

En cada vuelta se cronometra la lectura Parquet, la agrupación y el resultado
en pandas. Después se detiene el reloj y se comprueba equivalencia. La descarga,
conversión, ordenación de validación y exportación no entran en ese tiempo.

**Observa:** ¿gana siempre el mismo motor? La GPU puede ser más lenta si el
trabajo es pequeño o dominan lectura y transferencias. No borramos cachés del
sistema: las repeticiones no representan lecturas de disco frío. Registramos
hilos y configuración; este ejercicio no mide RAM pico ni VRAM pico.


In [ ]:
assert gpu_verificada, 'Primero ejecutar correctamente L8'
def ejecutar(motor, nombre):
    if motor == 'pandas_cpu':
        return analizar_pandas(CONSULTAS[nombre])
    engine = 'streaming' if motor == 'polars_cpu' else engine_gpu
    return consultas_polars[nombre].collect(engine=engine).to_pandas()

motores = ['pandas_cpu', 'polars_cpu', 'polars_gpu']
for nombre, claves in CONSULTAS.items():
    for motor in motores:
        calentamiento = ejecutar(motor, nombre)
        pd.testing.assert_frame_equal(comparable(referencias[nombre], claves), comparable(calentamiento, claves))
        del calentamiento

mediciones = []
for nombre, claves in CONSULTAS.items():
    for repeticion in range(REPETICIONES):
        desplazamiento = repeticion % len(motores)
        orden = motores[desplazamiento:] + motores[:desplazamiento]
        for motor in orden:
            inicio = perf_counter()
            resultado = ejecutar(motor, nombre)
            segundos = perf_counter() - inicio
            pd.testing.assert_frame_equal(comparable(referencias[nombre], claves), comparable(resultado, claves))
            assert int(resultado['reportes'].sum()) == TOTAL_ESPERADO
            mediciones.append({'consulta': nombre, 'motor': motor,
                'repeticion': repeticion + 1, 'segundos': segundos,
                'filas_entrada': TOTAL_ESPERADO, 'grupos': len(resultado), 'equivalente': True})
            print(nombre, repeticion + 1, motor, round(segundos, 4), 's', flush=True)
            del resultado


## L10. Mostrar nombres, construir rankings y comparar motores

La celda obtiene DIVIPOLA mediante `python 00_datos.py --descargar divipola`
y `python 00_datos.py --verificar divipola`, desde `kit`. Lee este CSV también
con Polars. Antes de unir nombres comprueba que el código municipal sea único.

1. Mostrar reportes con municipio reconocido, sin correspondencia o de otro tipo territorial.
2. Mostrar top municipal por periodo y departamentos representados en ese top.
3. Mostrar top municipal por cada periodo, grupo etario y género.
4. Construir los rankings desde pandas y desde los agregados GPU y comprobar igualdad.
5. Guardar CSV, tiempos y controles. Nombres y rankings se trabajan sobre tablas
   ya agregadas en CPU, fuera del benchmark. Los empates se resuelven por código.

Son volúmenes de reportes, no tasas por población ni riesgo municipal.


In [ ]:
subprocess.run([sys.executable, '00_datos.py', '--descargar', 'divipola'], cwd=KIT, check=True)
subprocess.run([sys.executable, '00_datos.py', '--verificar', 'divipola'], cwd=KIT, check=True)
catalogo = (pl.read_csv(KIT / 'data/raw/divipola.csv', infer_schema=False)
    .select(
        pl.col('Código Departamento').str.strip_chars().str.pad_start(2, '0').alias('codigo_departamento'),
        pl.col('Código Municipio').str.strip_chars().str.pad_start(5, '0').alias('codigo_municipio'),
        pl.col('Nombre Departamento').alias('departamento'),
        pl.col('Nombre Municipio').alias('municipio'),
        pl.col('Tipo: Municipio / Isla / Área no municipalizada').alias('tipo_territorio'))
    .to_pandas())
for columna in ['codigo_departamento', 'codigo_municipio']:
    catalogo[columna] = catalogo[columna].astype('string')
assert not catalogo.duplicated(['codigo_departamento', 'codigo_municipio']).any()
assert not catalogo.duplicated('codigo_municipio').any(), 'Resolver duplicados antes del join'

def con_nombres(conteos):
    tabla = conteos.copy()
    for col in ['codigo_departamento', 'codigo_municipio']:
        tabla[col] = tabla[col].astype('string')
    tabla = tabla.merge(catalogo, on=['codigo_departamento', 'codigo_municipio'],
                        how='left', validate='many_to_one', indicator=True)
    tabla['es_municipio'] = (tabla['_merge'].eq('both') &
        tabla['tipo_territorio'].str.strip().str.match(r'(?i)^municipio(?:\s|$)', na=False))
    tabla['cobertura'] = 'sin correspondencia'
    tabla.loc[tabla['_merge'].eq('both'), 'cobertura'] = 'otro tipo de territorio'
    tabla.loc[tabla['es_municipio'], 'cobertura'] = 'municipio reconocido'
    assert len(tabla) == len(conteos), 'El join multiplicó grupos'
    return tabla.drop(columns='_merge')

def ranking(conteos, estratos):
    tabla = con_nombres(conteos)
    tabla = tabla.loc[tabla['es_municipio']].copy()
    tabla = tabla.sort_values(estratos + ['reportes', 'codigo_departamento', 'codigo_municipio'],
        ascending=[True] * len(estratos) + [False, True, True])
    tabla['posicion'] = tabla.groupby(estratos, dropna=False).cumcount() + 1
    return tabla.loc[tabla['posicion'] <= TOP_N,
        estratos + ['posicion', 'codigo_departamento', 'departamento',
                    'codigo_municipio', 'municipio', 'reportes']].reset_index(drop=True)

for nombre, resultado in referencias.items():
    resultado.to_csv(OUT / f'{nombre}_verificado.csv', index=False)
por_corte = referencias['municipios_periodo'].groupby('corte_archivo')['reportes'].sum()
assert por_corte.to_dict() == {f['periodo']: f['filas'] for f in FUENTES}
municipios_con_nombres = con_nombres(referencias['municipios_periodo'])
cobertura = municipios_con_nombres.groupby(['corte_archivo', 'cobertura'], as_index=False)['reportes'].sum()
display(cobertura)
cobertura.to_csv(OUT / 'cobertura_municipal.csv', index=False)
municipios_con_nombres.loc[~municipios_con_nombres['es_municipio']].to_csv(
    OUT / 'reportes_fuera_ranking_municipal.csv', index=False)

# Ejercicio 1: municipios con más quejas y sus departamentos, por periodo.
top_municipios = ranking(referencias['municipios_periodo'], ['corte_archivo'])
top_municipios_gpu = ranking(municipios_gpu, ['corte_archivo'])
pd.testing.assert_frame_equal(top_municipios, top_municipios_gpu, check_dtype=False)
departamentos_top = top_municipios.groupby(
    ['corte_archivo', 'codigo_departamento', 'departamento'], as_index=False).agg(
        municipios_en_top=('codigo_municipio', 'nunique'),
        reportes_en_municipios_top=('reportes', 'sum'))
display(top_municipios)
display(departamentos_top)
top_municipios.to_csv(OUT / 'top_municipios_por_periodo.csv', index=False)
departamentos_top.to_csv(OUT / 'departamentos_del_top_municipal.csv', index=False)

# Ejercicio 2: top separado por periodo, grupo etario y género.
estratos = ['corte_archivo', 'grupo_etario', 'genero']
top_demografico = ranking(referencias['municipios_edad_genero'], estratos)
top_demografico_gpu = ranking(demografia_gpu, estratos)
pd.testing.assert_frame_equal(top_demografico, top_demografico_gpu, check_dtype=False)
display(top_demografico)
top_demografico.to_csv(OUT / 'top_municipios_por_edad_genero_periodo.csv', index=False)
print('Ambos rankings coinciden desde los agregados pandas y GPU')

tiempos = pd.DataFrame(mediciones)
assert len(tiempos) == len(CONSULTAS) * len(motores) * REPETICIONES
resumen = tiempos.groupby(['consulta', 'motor'], as_index=False).agg(
    mediana_s=('segundos', 'median'), min_s=('segundos', 'min'), max_s=('segundos', 'max'))
comparacion = resumen.pivot(index='consulta', columns='motor', values='mediana_s')
comparacion['pandas_sobre_gpu'] = comparacion['pandas_cpu'] / comparacion['polars_gpu']
comparacion['polars_cpu_sobre_gpu'] = comparacion['polars_cpu'] / comparacion['polars_gpu']
display(resumen)
display(comparacion)

versiones = {}
for paquete in ['pandas', 'polars', 'pyarrow', 'cudf-polars-cu12', 'cudf-polars-cu13']:
    try:
        versiones[paquete] = metadata.version(paquete)
    except metadata.PackageNotFoundError:
        versiones[paquete] = None
control_lab = {'periodos': PERIODOS, 'reportes': TOTAL_ESPERADO,
    'descargas': descargas, 'conversiones': conversiones,
    'gpu_ejecutada_y_equivalente': gpu_verificada,
    'dispositivo_cuda': DISPOSITIVO_CUDA, 'entorno_gpu_verificado_previamente': True,
    'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES'),
    'versiones': versiones, 'python': sys.version, 'plataforma': platform.platform(),
    'cpu_logicas': os.cpu_count(), 'hilos_polars': pl.thread_pool_size(),
    'bloque_analisis_pandas': BLOQUE, 'repeticiones': REPETICIONES,
    'top_n': TOP_N, 'ubicacion': 'persona afectada',
    'motor_conversion_csv_parquet': 'polars streaming CPU',
    'criterio_edad': '0–5, 6–11, 12–17, 18–28, 29–59, 60 o más; válido 0–120',
    'cobertura_municipal': cobertura.to_dict(orient='records'),
    'frontera': 'Parquet → agregación → resultado pandas en CPU',
    'descarga_y_conversion_incluidas_en_benchmark': False,
    'ram_pico_bytes': None, 'vram_pico_bytes': None}
tiempos.to_csv(OUT / 'benchmark_pandas_polars_gpu.csv', index=False)
pd.DataFrame(primeras).to_csv(OUT / 'primeras_ejecuciones.csv', index=False)
resumen.to_csv(OUT / 'resumen_benchmark.csv', index=False)
comparacion.to_csv(OUT / 'comparacion_medianas_y_factores.csv')
(OUT / 'control_laboratorio.json').write_text(json.dumps(control_lab, ensure_ascii=False, indent=2))
print('Evidencias:', OUT)


## L11. Explicar y entregar

1. Mostrar los tres CSV y los tres Parquet; explicar `scan_csv`, `with_columns`
   y `sink_parquet`. No se usa pandas para leer ni convertir CSV en este laboratorio.
2. Explicar ubicación de persona afectada, grupos etarios y datos no informados.
3. Entregar top municipal por periodo y departamentos representados.
4. Entregar top municipal por cada periodo, grupo etario y género.
5. Mostrar cobertura y reportes fuera del ranking por ubicación o tipo territorial.
6. Demostrar igualdad de grupos y rankings pandas/GPU.
7. Comparar medianas y dispersión de pandas CPU, Polars CPU y Polars GPU.
8. Distinguir conversión Polars CPU, agregación GPU y presentación de rankings CPU.

Adjuntar CSV y `control_laboratorio.json` al informe único de P04.
No interpretar conteos como tasas de riesgo. Si GPU falla, informar el error;
no presentar una ejecución CPU como si hubiera sido GPU.
